In [1]:
import pandas as pd
from pathlib import Path

In [2]:
ventas = pd.read_csv(Path(r"E:\ProyectoAnalisisElectrico\TransferenciasEconomicas\Ventas_SEN_2505_2604.csv"), sep=",")
calor = pd.read_csv(Path(r"E:\ProyectoAnalisisElectrico\CupraThermV2\Resumen_Establecimientos.csv"), sep=",")

In [3]:
ventas.head()

,AÑO,MES,RECONOCE EL RETIRO EN BALANCES DE TRANSFERENCIAS_O,CLIENTE SUMINISTRADO POR_O,CLIENTE2,RUTCLIENTE,TIPO,REGION,SECTOR,SUBSECTOR,...,ENERGIA2,RECONOCE EL RETIRO EN BALANCES DE TRANSFERENCIAS,CLIENTE SUMINISTRADO POR,RUTCLIENTEPUNTO DE CONEXIÓN,CLIENTE,RETIRO,clave,ID_PC,col1,col2
0,2026,1,ACCIONA ENERGIA CHILE HOLDINGS S.A.,ACCIONA ENERGIA CHILE HOLDINGS S.A.,AGUAS DEL ALTIPLANO S.A.,76215634-2,L,Tarapacá,Industrial,Agua,...,1202.030146,ACCIONA ENERGÍA CHILE HOLDINGS S.A.,ACCIONA ENERGÍA CHILE HOLDINGS S.A.,76215634-2Pozo Almonte 023,AGUAS DEL ALTIPLANO S.A.,1202.030146,A.ALTIPLANO,9278.0,NaN,NaN
1,2026,1,ACCIONA ENERGIA CHILE HOLDINGS S.A.,ACCIONA ENERGIA CHILE HOLDINGS S.A.,AGUAS DEL ALTIPLANO S.A.,76215634-2,L,Tarapacá,Industrial,Agua,...,1681.772740,ACCIONA ENERGÍA CHILE HOLDINGS S.A.,ACCIONA ENERGÍA CHILE HOLDINGS S.A.,76215634-2Tamarugal 023,AGUAS DEL ALTIPLANO S.A.,1681.772740,TAMARUGL_023_E5_ECL,9278.0,NaN,NaN
2,2026,1,ACCIONA ENERGIA CHILE HOLDINGS S.A.,ACCIONA ENERGIA CHILE HOLDINGS S.A.,Compañía Exploradora y Explotadora Minera Chil...,82789400-1,L,Atacama,Minero,Minas Varias,...,6324.759785,ACCIONA ENERGÍA CHILE HOLDINGS S.A.,ACCIONA ENERGÍA CHILE HOLDINGS S.A.,82789400-1Cerrillos 023,Compañía Exploradora y Explotadora Minera Chil...,6324.759785,COEMIN,9615.0,NaN,NaN
3,2026,1,ACCIONA ENERGIA CHILE HOLDINGS S.A.,ACCIONA ENERGIA CHILE HOLDINGS S.A.,EMPRESA NACIONAL DE MINERIA,61703000-4,L,Atacama,Minero,Minas Varias,...,3975.100920,ACCIONA ENERGÍA CHILE HOLDINGS S.A.,ACCIONA ENERGÍA CHILE HOLDINGS S.A.,61703000-4Cardones 110,EMPRESA NACIONAL DE MINERIA,3975.100920,L_EN005,3626.0,NaN,NaN
4,2026,1,ACCIONA ENERGIA CHILE HOLDINGS S.A.,ACCIONA ENERGIA CHILE HOLDINGS S.A.,NUEVA ATACAMA S.A.,76850128-9,L,Atacama,Industrial,Industrias Varias,...,1738.227626,ACCIONA ENERGÍA CHILE HOLDINGS S.A.,ACCIONA ENERGÍA CHILE HOLDINGS S.A.,76850128-9Caldera 110,NUEVA ATACAMA S.A.,1738.227626,L_ECONSSA_COPIAPO,13251.0,NaN,NaN


In [4]:
ventas["REGION"].unique() 

<ArrowStringArray>
[                     'Tarapacá',                       'Atacama',
                   'Antofagasta',                      'Coquimbo',
     'Metropolitana De Santiago',                       'Bío Bío',
                         'Ñuble',                      'Los Ríos',
                        'Biobío',                         'Maule',
                  'La Araucania',                    'Valparaíso',
 'Libertador Bernardo O'Higgins',                  'La Araucanía',
            'Arica y Parinacota',                     'Los Lagos',
                 'La Araucania ']
Length: 17, dtype: str

In [5]:
calor["REGION"].unique()

<ArrowStringArray>
[              'Metropolitana de Santiago',
                               'Los Lagos',
                             'Antofagasta',
                                'Tarapacá',
     'Libertador Gral. Bernardo O'Higgins',
                              'Valparaíso',
                                   'Maule',
                                   'Ñuble',
                                  'Biobío',
                               'Araucanía',
                      'Arica y Parinacota',
    'Magallanes y de la Antártica Chilena',
                                'Coquimbo',
                                'Los Ríos',
                                 'Atacama',
 'Aysén del Gral. Carlos Ibañez del Campo']
Length: 16, dtype: str

In [6]:
mapeo_regiones = {
    "Metropolitana De Santiago": "Metropolitana de Santiago",
    "Libertad Bernardo O'Higgins": "Libertad Gral. Bernardo O'Higgins",
    "La Araucania": "Araucanía",
    "La Araucanía": "Araucanía",
    "Bío Bío": "Biobío",
    "Biobío": "Biobío" 
}

ventas["REGION_HOMOLOGADA"] = ventas["REGION"].replace(mapeo_regiones)

In [7]:
agregaciones_calor = {
    'DEMANDA_CALOR_MWH': ['sum', 'mean', 'std', 'max', 'min'],
    'NOMBRE_ESTABLECIMIENTO': lambda x: list(set(x.dropna())),
    'COMBUSTIBLE_PRIMARIO': lambda x: list(set(x.dropna())),
    'RUBRO': lambda x: list(set(x.dropna()))
}

df_calor_hubs = calor.groupby(
    ['RUT_RAZON_SOCIAL', 'REGION']
).agg(agregaciones_calor).reset_index()

df_calor_hubs.columns = [
    '_'.join(col).strip('_') if type(col) is tuple else col 
    for col in df_calor_hubs.columns.values
]

df_calor_hubs = df_calor_hubs.rename(columns={
    'NOMBRE_ESTABLECIMIENTO_<lambda>': 'NOMBRE_ESTABLECIMIENTO',
    'COMBUSTIBLE_PRIMARIO_<lambda>': 'COMBUSTIBLE_PRIMARIO',
    'RUBRO_<lambda>': 'RUBRO'
})

columnas_texto = ['NOMBRE_ESTABLECIMIENTO', 'COMBUSTIBLE_PRIMARIO', 'RUBRO']

for col in columnas_texto:
    df_calor_hubs[col] = df_calor_hubs[col].apply(lambda x: '::'.join(map(str, x)))

df_calor_hubs['DEMANDA_CALOR_MWH_std'] = df_calor_hubs['DEMANDA_CALOR_MWH_std'].fillna(0)

In [8]:
cruce_calor_ventas = pd.merge(df_calor_hubs, ventas, how="inner", left_on=["RUT_RAZON_SOCIAL", "REGION"], right_on=["RUTCLIENTE", "REGION_HOMOLOGADA"])

In [9]:
cruce_calor_ventas.head()

,RUT_RAZON_SOCIAL,REGION_x,DEMANDA_CALOR_MWH_sum,DEMANDA_CALOR_MWH_mean,DEMANDA_CALOR_MWH_std,DEMANDA_CALOR_MWH_max,DEMANDA_CALOR_MWH_min,NOMBRE_ESTABLECIMIENTO,COMBUSTIBLE_PRIMARIO,RUBRO,...,RECONOCE EL RETIRO EN BALANCES DE TRANSFERENCIAS,CLIENTE SUMINISTRADO POR,RUTCLIENTEPUNTO DE CONEXIÓN,CLIENTE,RETIRO,clave,ID_PC,col1,col2,REGION_HOMOLOGADA
0,59087530-9,Antofagasta,34205.48312,34205.48312,0.0,34205.48312,34205.48312,PLANTA ECOMETALES,Petróleo N 6,Gestores de residuos,...,ENEL GENERACION CHILE S.A.,ENEL GENERACION CHILE S.A.,59087530-9KM6 100,Ecometales Ltd.,633.806476,21334709000RC,12990.0,NaN,NaN,Antofagasta
1,59087530-9,Antofagasta,34205.48312,34205.48312,0.0,34205.48312,34205.48312,PLANTA ECOMETALES,Petróleo N 6,Gestores de residuos,...,ENEL GENERACION CHILE S.A.,ENEL GENERACION CHILE S.A.,59087530-9KM6 100,Ecometales Ltd.,599.088531,21334709000RC,12990.0,NaN,NaN,Antofagasta
2,59087530-9,Antofagasta,34205.48312,34205.48312,0.0,34205.48312,34205.48312,PLANTA ECOMETALES,Petróleo N 6,Gestores de residuos,...,ENEL GENERACION CHILE S.A.,ENEL GENERACION CHILE S.A.,59087530-9KM6 100,Ecometales Ltd.,607.312339,21334709000RC,12990.0,NaN,NaN,Antofagasta
3,59087530-9,Antofagasta,34205.48312,34205.48312,0.0,34205.48312,34205.48312,PLANTA ECOMETALES,Petróleo N 6,Gestores de residuos,...,ENEL GENERACION CHILE S.A.,ENEL GENERACION CHILE S.A.,59087530-9KM6 100,Ecometales Ltd.,593.490651,21334709000RC,12990.0,NaN,NaN,Antofagasta
4,59087530-9,Antofagasta,34205.48312,34205.48312,0.0,34205.48312,34205.48312,PLANTA ECOMETALES,Petróleo N 6,Gestores de residuos,...,ENEL GENERACION CHILE S.A.,ENEL GENERACION CHILE S.A.,59087530-9Chuquicamata 100,Ecometales Ltd.,457.057146,21334709000RC,12990.0,NaN,NaN,Antofagasta


In [10]:
cruce_calor_ventas.columns

Index(['RUT_RAZON_SOCIAL', 'REGION_x', 'DEMANDA_CALOR_MWH_sum',
       'DEMANDA_CALOR_MWH_mean', 'DEMANDA_CALOR_MWH_std',
       'DEMANDA_CALOR_MWH_max', 'DEMANDA_CALOR_MWH_min',
       'NOMBRE_ESTABLECIMIENTO', 'COMBUSTIBLE_PRIMARIO', 'RUBRO', 'AÑO', 'MES',
       'RECONOCE EL RETIRO EN BALANCES DE TRANSFERENCIAS_O',
       'CLIENTE SUMINISTRADO POR_O', 'CLIENTE2', 'RUTCLIENTE', 'TIPO',
       'REGION_y', 'SECTOR', 'SUBSECTOR', 'PUNTO DE CONEXIÓN', 'ENERGIA2',
       'RECONOCE EL RETIRO EN BALANCES DE TRANSFERENCIAS',
       'CLIENTE SUMINISTRADO POR', 'RUTCLIENTEPUNTO DE CONEXIÓN', 'CLIENTE',
       'RETIRO', 'clave', 'ID_PC', 'col1', 'col2', 'REGION_HOMOLOGADA'],
      dtype='str')

In [11]:
# Definimos la lista con el orden deseado adaptado a tu DataFrame actual
nuevo_orden = [
    # 1. Las columnas prioritarias
    'CLIENTE2', 
    'RUTCLIENTE', 
    'TIPO', 
    'clave', 
    'NOMBRE_ESTABLECIMIENTO', 
    'REGION_HOMOLOGADA', 
    'COMBUSTIBLE_PRIMARIO', 
    'AÑO', 
    'MES',
    
    # 2. Identificadores y datos generales (los que sobrevivieron a la agrupación)
    'RUT_RAZON_SOCIAL', 
    'RUBRO', 
    
    # 3. Ubicación geográfica
    'REGION_x', # Región original de la base de calor
    'REGION_y', # Región original de la base de ventas
    
    # 4. Detalles de cliente, sectores y transferencias
    'CLIENTE', 
    'SECTOR', 
    'SUBSECTOR', 
    'PUNTO DE CONEXIÓN', 
    'ID_PC',
    'RUTCLIENTEPUNTO DE CONEXIÓN', 
    'CLIENTE SUMINISTRADO POR_O', 
    'CLIENTE SUMINISTRADO POR', 
    'RECONOCE EL RETIRO EN BALANCES DE TRANSFERENCIAS_O', 
    'RECONOCE EL RETIRO EN BALANCES DE TRANSFERENCIAS', 
    'RETIRO', 
    
    # 5. Energía y variables numéricas (ahora incluye tus estadísticas de calor)
    'ENERGIA2',
    'DEMANDA_CALOR_MWH_sum', 
    'DEMANDA_CALOR_MWH_mean', 
    'DEMANDA_CALOR_MWH_std', 
    'DEMANDA_CALOR_MWH_max', 
    'DEMANDA_CALOR_MWH_min',
    
    # 6. Columnas misceláneas
    'col1', 
    'col2'
]

# Reasignamos el DataFrame para aplicar el orden
cruce_calor_ventas = cruce_calor_ventas[nuevo_orden]


In [12]:
# 1. Creamos la nueva columna 'PERIODO' con el formato YYMM
# Nos aseguramos de tratar AÑO y MES como enteros primero para evitar decimales indeseados (.0)
año_str = cruce_calor_ventas['AÑO'].astype(int).astype(str).str[-2:]
mes_str = cruce_calor_ventas['MES'].astype(int).astype(str).str.zfill(2)

cruce_calor_ventas['PERIODO'] = año_str + mes_str

# 2. Definimos la lista exacta con las columnas que solicitaste
columnas_finales = [
    'clave', 
    'CLIENTE', 
    'TIPO', 
    'NOMBRE_ESTABLECIMIENTO', 
    'DEMANDA_CALOR_MWH_sum', 
    'DEMANDA_CALOR_MWH_mean', 
    'DEMANDA_CALOR_MWH_std', 
    'DEMANDA_CALOR_MWH_max', 
    'DEMANDA_CALOR_MWH_min', 
    'RUT_RAZON_SOCIAL', 
    'REGION_HOMOLOGADA', 
    'PERIODO', # Reemplaza a AÑO y MES
    'RUBRO',
    'SECTOR', 
    'SUBSECTOR'
]

# 3. Filtramos el DataFrame para quedarnos solo con lo necesario
cruce_calor_ventas_limpio = cruce_calor_ventas[columnas_finales].copy()

# Imprimimos para verificar
print(cruce_calor_ventas_limpio.head())

           clave          CLIENTE TIPO NOMBRE_ESTABLECIMIENTO  \
0  21334709000RC  Ecometales Ltd.    L      PLANTA ECOMETALES   
1  21334709000RC  Ecometales Ltd.    L      PLANTA ECOMETALES   
2  21334709000RC  Ecometales Ltd.    L      PLANTA ECOMETALES   
3  21334709000RC  Ecometales Ltd.    L      PLANTA ECOMETALES   
4  21334709000RC  Ecometales Ltd.    L      PLANTA ECOMETALES   

   DEMANDA_CALOR_MWH_sum  DEMANDA_CALOR_MWH_mean  DEMANDA_CALOR_MWH_std  \
0            34205.48312             34205.48312                    0.0   
1            34205.48312             34205.48312                    0.0   
2            34205.48312             34205.48312                    0.0   
3            34205.48312             34205.48312                    0.0   
4            34205.48312             34205.48312                    0.0   

   DEMANDA_CALOR_MWH_max  DEMANDA_CALOR_MWH_min RUT_RAZON_SOCIAL  \
0            34205.48312            34205.48312       59087530-9   
1            34205.483

In [13]:
cruce_calor_ventas_limpio.rename(columns={"REGION_HOMOLOGADA": "REGION"}, inplace=True)

In [14]:
# 1. Definimos el diccionario corregido
macrozonas = {
    # 1.- Macrozona Norte Grande
    'Arica y Parinacota': 'Norte Grande',
    'Tarapacá': 'Norte Grande',
    'Antofagasta': 'Norte Grande',

    # 2.- Macrozona Norte Chico
    'Atacama': 'Norte Chico',
    'Coquimbo': 'Norte Chico',

    # 3.- Macrozona Centro
    'Valparaíso': 'Centro',
    'Metropolitana de Santiago': 'Centro', 

    # 4.- Macrozona Centro Sur
    'Libertador Gral. Bernardo O\'Higgins': 'Centro Sur',
    'Maule': 'Centro Sur',
    'Ñuble': 'Centro Sur',
    'Biobío': 'Centro Sur',

    # 5.- Macrozona Sur
    'Araucanía': 'Sur',  # <-- CORREGIDO: Ajustado a cómo viene en tu DF
    'Los Ríos': 'Sur',
    'Los Lagos': 'Sur',

    # 6.- Macrozona Austral
    'Aysén del General Carlos Ibáñez del Campo': 'Austral',
    'Magallanes y de la Antártica Chilena': 'Austral'
}

cruce_calor_ventas_limpio['macrozona'] = cruce_calor_ventas_limpio['REGION'].map(macrozonas)

In [15]:
cruce_calor_ventas_limpio.to_csv(Path(r"E:\ProyectoAnalisisElectrico\PotencialesClientes\CalorVentasRegionales.csv"), index=False)

In [16]:
cruce_calor_ventas_limpio.head()

,clave,CLIENTE,TIPO,NOMBRE_ESTABLECIMIENTO,DEMANDA_CALOR_MWH_sum,DEMANDA_CALOR_MWH_mean,DEMANDA_CALOR_MWH_std,DEMANDA_CALOR_MWH_max,DEMANDA_CALOR_MWH_min,RUT_RAZON_SOCIAL,REGION,PERIODO,RUBRO,SECTOR,SUBSECTOR,macrozona
0,21334709000RC,Ecometales Ltd.,L,PLANTA ECOMETALES,34205.48312,34205.48312,0.0,34205.48312,34205.48312,59087530-9,Antofagasta,2601,Gestores de residuos,Industrial,Industrias Varias,Norte Grande
1,21334709000RC,Ecometales Ltd.,L,PLANTA ECOMETALES,34205.48312,34205.48312,0.0,34205.48312,34205.48312,59087530-9,Antofagasta,2602,Gestores de residuos,Industrial,Industrias Varias,Norte Grande
2,21334709000RC,Ecometales Ltd.,L,PLANTA ECOMETALES,34205.48312,34205.48312,0.0,34205.48312,34205.48312,59087530-9,Antofagasta,2603,Gestores de residuos,Industrial,Industrias Varias,Norte Grande
3,21334709000RC,Ecometales Ltd.,L,PLANTA ECOMETALES,34205.48312,34205.48312,0.0,34205.48312,34205.48312,59087530-9,Antofagasta,2604,Gestores de residuos,Industrial,Industrias Varias,Norte Grande
4,21334709000RC,Ecometales Ltd.,L,PLANTA ECOMETALES,34205.48312,34205.48312,0.0,34205.48312,34205.48312,59087530-9,Antofagasta,2505,Gestores de residuos,Industrial,Industrias Varias,Norte Grande
